In [ ]:
pip install fastf1

In [ ]:
import fastf1 as ff1
import pandas as pd
import numpy as np
from fastf1 import plotting
from sklearn.preprocessing import StandardScaler

In [ ]:
import os

# Ensure the cache directory exists
if not os.path.exists('cache'):
    os.makedirs('cache')

ff1.Cache.enable_cache('cache')

In [ ]:
# Configura el plotting de FastF1
plotting.setup_mpl()

# --- 1. Carga de Datos ---

session = ff1.get_session(2025, 'Italian Grand Prix', 'R')
session.load()
print(f"Sesión cargada: {session.event.EventName} {session.name}")

In [ ]:
# --- 2. Definición de Zonas de Interés (ZOI) ---
# Definimos zonas de análisis con sub-partes: 'braking', 'apex', 'exit'
circuit_info = session.get_circuit_info()
analysis_zones = {
    'T1_Rettifilo': {
        'braking': (circuit_info.corners.loc[0]['Distance'] - 350, circuit_info.corners.loc[0]['Distance']),
        'apex': (circuit_info.corners.loc[0]['Distance'] - 50, circuit_info.corners.loc[1]['Distance'] + 50),
        'exit': (circuit_info.corners.loc[1]['Distance'], circuit_info.corners.loc[1]['Distance'] + 250)
    },
    'T4_Roggia': {
        'braking': (circuit_info.corners.loc[3]['Distance'] - 300, circuit_info.corners.loc[3]['Distance']),
        'apex': (circuit_info.corners.loc[3]['Distance'] - 50, circuit_info.corners.loc[4]['Distance'] + 50),
        'exit': (circuit_info.corners.loc[4]['Distance'], circuit_info.corners.loc[4]['Distance'] + 250)
    },
    'T11_Alboreto': {
        'braking': (circuit_info.corners.loc[10]['Distance'] - 350, circuit_info.corners.loc[10]['Distance']),
        'apex': (circuit_info.corners.loc[10]['Distance'] - 100, circuit_info.corners.loc[10]['Distance'] + 100),
        'exit': (circuit_info.corners.loc[10]['Distance'] + 100, circuit_info.corners.loc[10]['Distance'] + 350)
    }
}
print(f"Definidas {len(analysis_zones)} zonas de análisis.")

In [ ]:
# --- 3. Funciones de Feature Engineering ---

def get_braking_point(telemetry_fragment):
    """Devuelve la distancia donde 'Brake' se activa."""
    try:
        braking_start_row = telemetry_fragment[telemetry_fragment['Brake'] > 0.1].first_valid_index()
        if braking_start_row is not None:
            return telemetry_fragment.loc[braking_start_row, 'Distance']
    except Exception as e:
        pass
    return np.nan

def get_min_apex_speed(telemetry_fragment):
    """Devuelve la velocidad mínima en el fragmento de vértice."""
    try:
        if not telemetry_fragment.empty:
            return telemetry_fragment['Speed'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_deceleration(telemetry_fragment):
    """Devuelve la desaceleración máxima (en m/s^2) mientras se frena."""
    try:
        # Filtra solo los momentos en que el freno está aplicado
        braking_fragment = telemetry_fragment[telemetry_fragment['Brake'] > 0.1]
        if not braking_fragment.empty:
            # Encuentra el valor mínimo de aceleración (máxima desaceleración)
            return braking_fragment['Acceleration'].min()
    except Exception as e:
        pass
    return np.nan

def get_max_throttle_rate(telemetry_fragment):
    """Devuelve la tasa máxima de aplicación del acelerador (% por segundo)."""
    try:
        if not telemetry_fragment.empty:
            # Tasa de cambio del acelerador (0-100) por segundo
            throttle_rate = telemetry_fragment['Throttle'].diff() / telemetry_fragment['TimeDelta']

            # Encuentra la tasa positiva máxima
            max_rate = throttle_rate[throttle_rate > 0].max()

            if pd.notna(max_rate) and not np.isinf(max_rate):
                return max_rate
    except Exception as e:
        pass
    return np.nan


In [ ]:
# --- 4. Iteración por todos los pilotos y curvas (Multivariable) ---

driver_list = session.drivers
print(f"Analizando {len(driver_list)} pilotos...")

all_driver_features = []

for driver_code in driver_list:
    lap = session.laps.pick_drivers(driver_code).pick_fastest()

    if not isinstance(lap, pd.Series) or lap.empty:
        # print(f"Saltando a {driver_code} (sin vuelta rápida válida)")
        continue

    telemetry = lap.get_car_data().add_distance()

    # --- Pre-cálculo de Aceleración y TimeDelta (¡IMPORTANTE!) ---
    telemetry['Speed_ms'] = telemetry['Speed'] / 3.6  # Convertir km/h a m/s
    telemetry['TimeDelta'] = telemetry['Time'].diff().dt.total_seconds()
    telemetry['SpeedDelta'] = telemetry['Speed_ms'].diff()
    # Aceleración en m/s^2
    telemetry['Acceleration'] = telemetry['SpeedDelta'] / telemetry['TimeDelta']

    driver_data = {'Driver': driver_code}

    for zone_name, zone_parts in analysis_zones.items():

        # --- 1. Fragmentar la telemetría según las sub-zonas ---
        braking_fragment = telemetry.loc[
            (telemetry['Distance'] >= zone_parts['braking'][0]) & (telemetry['Distance'] <= zone_parts['braking'][1])
        ]
        apex_fragment = telemetry.loc[
            (telemetry['Distance'] >= zone_parts['apex'][0]) & (telemetry['Distance'] <= zone_parts['apex'][1])
        ]
        exit_fragment = telemetry.loc[
            (telemetry['Distance'] >= zone_parts['exit'][0]) & (telemetry['Distance'] <= zone_parts['exit'][1])
        ]

        # --- 2. Calcular y guardar cada feature ---
        driver_data[f'{zone_name}_BrakingPoint'] = get_braking_point(braking_fragment)
        driver_data[f'{zone_name}_MaxDecel'] = get_max_deceleration(braking_fragment)
        driver_data[f'{zone_name}_ApexSpeed'] = get_min_apex_speed(apex_fragment)
        driver_data[f'{zone_name}_ThrottleRate'] = get_max_throttle_rate(exit_fragment)

    all_driver_features.append(driver_data)

In [ ]:
# --- 5. Creación y Normalización del DataFrame ---

# Convertimos la lista de diccionarios en un DataFrame y eliminamos pilotos sin datos
df_raw_features = pd.DataFrame(all_driver_features).set_index('Driver').dropna()

display("\n--- DataFrame de Features (Datos Brutos) ---")
display(df_raw_features.head())

# Calculamos la media de cada feature (columna)
feature_averages = df_raw_features.mean()

# NORMALIZACIÓN: Restamos la media de la feature a cada piloto
df_normalized = df_raw_features - feature_averages

display("\n--- DataFrame Normalizado (Puntuación de Estilo) ---")
display(df_normalized.head())
display("\nInterpretación del DataFrame Normalizado:")
display("Valores positivos = 'más que el promedio' (ej. frenó más tarde, fue más rápido en vértice, etc.)")
display("Valores negativos = 'menos que el promedio'")
